In [ ]:
# Bench60 ladder @ 1M — HIGH_SPEEDUP path — CONFIG
# Edit ONLY CHUNK_INDEX if needed (pre-set per file).
#
# HIGH_SPEEDUP in this stack = N_WORKERS="auto" + ENGINE="hcompact"
# (same shape as greedy_baseline HIGH_SPEEDUP; there is no separate boolean).
# On ~51 GB / 8-core Colab, _resolve_workers memory-caps to ~6 workers at 1M
# (~7.6 GB/search) — never pin N_WORKERS=8 at 1M (would exceed 50 GB).
#
# Dataset = bench66[:60] = difficulty-stratified ladder (efficiency A/B).
# Mini-verified @ budget 1000: 4 workers → 2.84× wall vs serial (result-neutral).

REPO_URL   = "https://github.com/Avi161/ACSolverX.git"
REPO_DIR   = "ACSolverX"
BRANCH     = "cursor/heur-12h-anti-overfit-a42e"
CLONE      = True
UPDATE_REPO = True

MOUNT_DRIVE = True
DRIVE_DIR   = "/content/drive/MyDrive/acsolverx/hsearch_bench60_1m"

CHUNK_INDEX = 1

# ---- HIGH_SPEEDUP knobs (required for multi-worker under 50 GB) ------------
# ENGINE must be hcompact at 1M; hsolve+keep_path ≈34 GB/search → 1 worker.
# N_WORKERS="auto" = min(cores, floor((MemAvailable-2)/GB_per_search)).
cfg = dict(
    DATASET   = "bench66",
    SUBSET    = 60,

    ARMS      = ['baseline', 's12', 's28', 's20_mk2', 's24_k1_mk2'],

    CHUNKS       = 5,
    CHUNK_INDEX  = CHUNK_INDEX,

    # HIGH_SPEEDUP path (greedy_baseline analogue — no HIGH_SPEEDUP bool here):
    ENGINE       = "hcompact",   # REQUIRED
    N_WORKERS    = "auto",       # 8 cores → memory-capped ~6 @1M/50GB
    KEEP_PATH    = True,

    NODE_BUDGET = 1_000_000,
    CHECKPOINTS = [1000, 5000, 10000, 25000, 50000, 100000, 250000, 500000, 1000000],
    MAX_RELATOR_LENGTH = 48,
    RESUME    = True,
    OUT_STEM  = "hsearch_bench60_1m",
    STAGE_DIR = "/content/hsearch_stage",
)

HEARTBEAT_SECS = 60
PROGRESS_SECS  = 300


In [ ]:
# ==================== SETUP (clone / pull / mount / purge) ================
import os, sys, subprocess, importlib

def sh(cmd):
    print("$", cmd)
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.stdout: print(p.stdout[-2000:])
    if p.returncode != 0 and p.stderr: print("STDERR:", p.stderr[-2000:])

try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False
print("Colab:", IN_COLAB)

if IN_COLAB:
    BASE = "/content"
    os.chdir(BASE)
    if not os.path.isdir(REPO_DIR):
        if CLONE:
            sh(f"git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}")
    elif UPDATE_REPO:
        sh(f"cd {REPO_DIR} && git fetch origin {BRANCH} && git reset --hard FETCH_HEAD")
    sh(f"cd {REPO_DIR} && git log -1 --oneline")
    sh("pip -q install numba numpy")
    if MOUNT_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        os.makedirs(DRIVE_DIR, exist_ok=True)
    os.makedirs(cfg.get("STAGE_DIR", "/content/hsearch_stage"), exist_ok=True)
    REPO_ROOT = os.path.join(BASE, REPO_DIR)
else:
    REPO_ROOT = os.getcwd()
    while REPO_ROOT != "/" and not (
        os.path.isdir(os.path.join(REPO_ROOT, "experiments"))
        and os.path.isdir(os.path.join(REPO_ROOT, "data"))
    ):
        REPO_ROOT = os.path.dirname(REPO_ROOT)

os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print("repo root:", REPO_ROOT)
print("BRANCH:", BRANCH, "ENGINE:", cfg["ENGINE"], "N_WORKERS:", cfg["N_WORKERS"])
assert cfg["ENGINE"] == "hcompact", "ENGINE=hcompact required for HIGH_SPEEDUP @1M"

for _m in [m for m in sys.modules if m == "experiments" or m.startswith("experiments.")]:
    del sys.modules[_m]
importlib.invalidate_caches()

from experiments.heuristic_search.core.hsolve import greedy_search_h
from experiments.heuristic_search.runners import run_s24_scale as _rs
_ = greedy_search_h("xyx", "yx", 20, max_relator_length=32,
                    config=_rs.run_ab.ARMS["s20"])
print("kernels warm — setup done")
print("tip: Runtime → Restart → Run All resumes (UPDATE_REPO + flock + RESUME)")
print("ARMS:", cfg["ARMS"])


In [ ]:
# ==================== RUN (HIGH_SPEEDUP multi-worker) =====================
from experiments.heuristic_search.runners.run_s24_scale import run_s24_scale
from experiments.heuristic_search.runners import run_ab as _ra
nw, per = _ra._resolve_workers(
    {"N_WORKERS": cfg["N_WORKERS"]}, cfg["NODE_BUDGET"],
    cfg["MAX_RELATOR_LENGTH"], cfg["ENGINE"], cfg["KEEP_PATH"])
print(f"HIGH_SPEEDUP resolve: N_WORKERS={cfg['N_WORKERS']} -> {nw} workers "
      f"(~{per:.1f} GB/search est., ENGINE={cfg['ENGINE']})")
print(f"budget={cfg['NODE_BUDGET']:,} chunk={cfg['CHUNK_INDEX']}/{cfg['CHUNKS']} "
      f"arms={cfg['ARMS']}")
run_s24_scale(
    cfg,
    out_dir=(DRIVE_DIR if (IN_COLAB and MOUNT_DRIVE) else "results/hsearch"),
    heartbeat_secs=HEARTBEAT_SECS,
    progress_secs=PROGRESS_SECS,
)
print("done — leave session up until jsonl mirror finishes.")
